# Group 8 — MCE 415: YOLOv10 Road Anomaly Detection
## Full Analysis Notebook — Kaggle Edition
**Run all cells top-to-bottom. All outputs are saved to `/kaggle/working/Group8_Results/`**

| Item | Value |
|------|-------|
| Model | YOLOv10-M (COCO pretrained) |
| Classes | Pothole, Speedbump, Crack |
| LRs Evaluated | 0.00001 · 0.0001 · 0.001 · 0.01 · 0.1 |
| Epochs | 100 |
| Optimizer | AdamW |


---
### Cell 1 — Install dependencies & verify GPU

In [ ]:
# CELL 1 — Install dependencies & verify GPU
import subprocess, sys

def pip_install(pkg):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"WARNING: pip install failed for {pkg}: {result.stderr[:200]}")
    return result.returncode == 0

pip_install("ultralytics>=8.2.50")
pip_install("pyyaml")
pip_install("seaborn")
pip_install("gdown")

import torch
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")
    print(f"VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING   : No GPU detected — inference will be slow.")

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device    : {DEVICE}")


---
### Cell 2 — Core imports & global configuration

In [ ]:
# CELL 2 — Core imports & global configuration
import os, yaml, shutil, time, platform, zipfile, glob, warnings, traceback
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from datetime import datetime
from io import StringIO

from ultralytics import YOLO

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.15)
plt.rcParams.update({
    "figure.dpi"    : 150,
    "savefig.dpi"   : 300,
    "savefig.bbox"  : "tight",
    "font.family"   : "DejaVu Sans",
    "axes.titlepad" : 14,
    "axes.labelpad" : 8,
})

# ── Global constants ────────────────────────────────────────
SEED           = 42
LEARNING_RATES = [0.00001, 0.0001, 0.001, 0.01, 0.1]
CLASS_NAMES    = ["Pothole", "Speedbump", "Crack"]
MODEL_WEIGHTS  = "yolov10m.pt"          # COCO pretrained — used for architecture info

LR_COLOURS = ["#1565C0", "#2E7D32", "#E65100", "#6A1B9A", "#B71C1C"]
LR_MARKERS = ["o",       "s",       "D",       "^",       "v"      ]
LR_LABELS  = [f"LR = {lr}" for lr in LEARNING_RATES]

OUTPUT_DIR  = Path("/kaggle/working/Group8_Results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Ultralytics : {__import__('ultralytics').__version__}")
print(f"Output dir  : {OUTPUT_DIR}")
print(f"Script start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


---
### Cell 3 — Download trained models from Google Drive

In [ ]:
# CELL 3 — Download trained models & map folder paths
# ────────────────────────────────────────────────────────────────────────────
# UPDATE GDRIVE_MODELS_ID below with the Google Drive file ID of your
# consolidated models zip (all 5 LR runs zipped together).
# Expected structure inside the zip:
#   Group8_LR_1e-05_results/lr_1e-05/weights/best.pt
#   Group8_LR_1e-05_results/lr_1e-05/results.csv
#   Group8_LR_0.0001_results/lr_0.0001/weights/best.pt   ... etc.
# ────────────────────────────────────────────────────────────────────────────

GDRIVE_MODELS_ID = "1xWuaaYg7hSHwnyGXEfeEL8ag6PoWGPrF"   # <-- YOUR FILE ID
MODELS_BASE      = Path("/kaggle/working/Downloaded Models")
MODELS_ZIP       = Path("/kaggle/working/Downloaded_Models.zip")

if not MODELS_BASE.exists():
    try:
        import gdown as _gdown
        print("Downloading trained models from Google Drive …")
        _gdown.download(
            f"https://drive.google.com/uc?id={GDRIVE_MODELS_ID}",
            str(MODELS_ZIP), quiet=False,
        )
        assert MODELS_ZIP.exists() and MODELS_ZIP.stat().st_size > 100_000, (
            "Download failed or file too small. Check Google Drive sharing (Anyone with link)."
        )
        print(f"Downloaded  : {MODELS_ZIP.stat().st_size / 1e6:.1f} MB")
        print("Extracting …")
        with zipfile.ZipFile(MODELS_ZIP, "r") as zf:
            zf.extractall("/kaggle/working")
        if MODELS_ZIP.exists():
            MODELS_ZIP.unlink()
        print(f"Extracted to: {MODELS_BASE}")
    except Exception as e:
        print(f"ERROR during model download: {e}")
        traceback.print_exc()
else:
    print(f"Models folder already present: {MODELS_BASE}")

# ── Folder map: lr value → run directory ───────────────────
LR_FOLDER_MAP = {
    0.00001 : MODELS_BASE / "Group8_LR_1e-05_results"  / "lr_1e-05",
    0.0001  : MODELS_BASE / "Group8_LR_0.0001_results" / "lr_0.0001",
    0.001   : MODELS_BASE / "Group8_LR_0.001_results"  / "lr_0.001",
    0.01    : MODELS_BASE / "Group8_LR_0.01_results"   / "lr_0.01",
    0.1     : MODELS_BASE / "Group8_LR_0.1_results"    / "lr_0.1",
}

print("\nVerifying trained model folders:")
available_lrs = []
for lr in LEARNING_RATES:
    d   = LR_FOLDER_MAP[lr]
    csv = (d / "results.csv").exists()
    wt  = (d / "weights" / "best.pt").exists()
    ok  = csv and wt
    tag = "✓  OK" if ok else "✗  MISSING"
    print(f"  LR = {lr:<10}  [{tag}]   csv={csv}  best.pt={wt}")
    print(f"              → {d}")
    if ok:
        available_lrs.append(lr)

if not available_lrs:
    print("\n⚠  No valid runs found. Listing MODELS_BASE contents for debugging:")
    if MODELS_BASE.exists():
        for p in sorted(MODELS_BASE.rglob("*"))[:60]:
            print(f"   {p}")
    else:
        print("   MODELS_BASE does not exist.")
        print("   Contents of /kaggle/working:")
        for p in sorted(Path("/kaggle/working").iterdir()):
            print(f"   {p}")
    raise RuntimeError(
        "No valid LR run folders found. "
        "Update LR_FOLDER_MAP paths or re-check your Google Drive zip structure."
    )

print(f"\n{len(available_lrs)}/{len(LEARNING_RATES)} runs ready: {available_lrs}")


---
### Cell 4 — Download & prepare dataset

In [ ]:
# CELL 4 — Download & prepare dataset (same source used during training)
import gdown

GDRIVE_FILE_ID = "18JDX57ppDZpPvWUaeFFkXgHM28NgIzV5"   # unified dataset zip
DATASET_ROOT   = Path("/kaggle/working/unified_dataset")
EXTRACT_TMP    = Path("/kaggle/working/_extract_tmp")
ZIP_PATH       = Path("/kaggle/working/unified_dataset.zip")
IMG_EXTS       = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}

already_extracted = (
    (DATASET_ROOT / "train" / "images").exists() or
    (DATASET_ROOT / "images" / "train").exists()
)

if not already_extracted:
    if not ZIP_PATH.exists():
        try:
            print("Downloading dataset …")
            gdown.download(id=GDRIVE_FILE_ID, output=str(ZIP_PATH), quiet=False)
            assert ZIP_PATH.stat().st_size > 1_000_000, "Dataset zip too small."
            print(f"Downloaded: {ZIP_PATH.stat().st_size / 1e6:.1f} MB")
        except Exception as e:
            raise RuntimeError(f"Dataset download failed: {e}")

    for d in [DATASET_ROOT, EXTRACT_TMP]:
        if d.exists():
            shutil.rmtree(d)
    EXTRACT_TMP.mkdir(parents=True)

    print("Extracting …")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(str(EXTRACT_TMP))

    cands_a = glob.glob(str(EXTRACT_TMP / "**" / "train" / "images"), recursive=True)
    cands_b = glob.glob(str(EXTRACT_TMP / "**" / "images" / "train"), recursive=True)
    if cands_a:
        actual_root = Path(cands_a[0]).parent.parent
        print(f"Layout A detected: {actual_root}")
    elif cands_b:
        actual_root = Path(cands_b[0]).parent.parent
        print(f"Layout B detected: {actual_root}")
    else:
        raise FileNotFoundError("Cannot locate train/images or images/train in dataset zip.")

    shutil.move(str(actual_root), str(DATASET_ROOT))
    if EXTRACT_TMP.exists():
        shutil.rmtree(EXTRACT_TMP)
    print(f"Dataset ready: {DATASET_ROOT}")
else:
    print(f"Dataset already present: {DATASET_ROOT}")

# ── Layout helpers ──────────────────────────────────────────
if (DATASET_ROOT / "train" / "images").exists():
    LAYOUT = "A"
    def img_dir(split): return DATASET_ROOT / split / "images"
    def lbl_dir(split): return DATASET_ROOT / split / "labels"
    yaml_train, yaml_val, yaml_test = "train/images", "val/images", "test/images"
elif (DATASET_ROOT / "images" / "train").exists():
    LAYOUT = "B"
    def img_dir(split): return DATASET_ROOT / "images" / split
    def lbl_dir(split): return DATASET_ROOT / "labels" / split
    yaml_train, yaml_val, yaml_test = "images/train", "images/val", "images/test"
else:
    raise RuntimeError(f"Unknown dataset layout in {DATASET_ROOT}")

print(f"Layout: {LAYOUT}")

# ── Write / update data.yaml ────────────────────────────────
DATA_YAML = DATASET_ROOT / "data.yaml"
data_cfg = {}
if DATA_YAML.exists():
    with open(DATA_YAML) as f:
        data_cfg = yaml.safe_load(f) or {}
data_cfg.update({
    "path" : str(DATASET_ROOT),
    "train": yaml_train,
    "val"  : yaml_val,
    "test" : yaml_test,
    "nc"   : 3,
    "names": CLASS_NAMES,
})
with open(DATA_YAML, "w") as f:
    yaml.dump(data_cfg, f, default_flow_style=False, sort_keys=False)

# ── Count images & annotations per split ───────────────────
print("\nDataset statistics:")
dataset_counts      = {}
dataset_annotations = {}
for split in ["train", "val", "test"]:
    d = img_dir(split)
    if d.exists():
        imgs = len([f for f in d.iterdir() if f.suffix.lower() in IMG_EXTS])
        dataset_counts[split] = imgs

        cls_counts = defaultdict(int)
        ld = lbl_dir(split)
        if ld.exists():
            for lf in ld.glob("*.txt"):
                with open(lf) as f:
                    for line in f:
                        if line.strip():
                            cls_counts[int(line.split()[0])] += 1
        dataset_annotations[split] = dict(cls_counts)
        ann_total = sum(cls_counts.values())
        print(f"  {split:>5}: {imgs:>5} images | {ann_total:>5} annotations")
        for cid in sorted(cls_counts):
            print(f"         Class {cid} ({CLASS_NAMES[cid]}): {cls_counts[cid]}")
    else:
        dataset_counts[split]      = 0
        dataset_annotations[split] = {}
        print(f"  {split:>5}: (not found)")

total_images = sum(dataset_counts.values())
print(f"\n  TOTAL: {total_images} images across all splits")
print("Dataset ready ✓")


---
### Cell 5 — Load all training results CSVs

In [ ]:
# CELL 5 — Load all training results CSVs & identify best LR
all_results    = {}
best_lr_per_run = {}

for lr in available_lrs:
    csv_path = LR_FOLDER_MAP[lr] / "results.csv"
    try:
        df = pd.read_csv(csv_path)
        df.columns = df.columns.str.strip()
        all_results[lr] = df
        print(f"  LR = {lr}: {len(df)} epochs loaded | columns: {list(df.columns)}")
    except Exception as e:
        print(f"  LR = {lr}: FAILED — {e}")

if not all_results:
    raise RuntimeError("No results CSVs loaded — cannot continue.")

for lr, df in all_results.items():
    best_idx = df["metrics/mAP50(B)"].idxmax()
    best_lr_per_run[lr] = {
        "epoch"     : int(df.loc[best_idx, "epoch"]) + 1,
        "mAP50"     : float(df.loc[best_idx, "metrics/mAP50(B)"]),
        "mAP50_95"  : float(df.loc[best_idx, "metrics/mAP50-95(B)"]),
        "precision" : float(df.loc[best_idx, "metrics/precision(B)"]),
        "recall"    : float(df.loc[best_idx, "metrics/recall(B)"]),
    }

BEST_LR = max(best_lr_per_run, key=lambda lr: best_lr_per_run[lr]["mAP50"])
print(f"\nBest LR : {BEST_LR}   (mAP@0.5 = {best_lr_per_run[BEST_LR]['mAP50']:.4f})")


---
### Cell 6 — Table 1: Model Architecture Summary (GFLOPs + Layer Count)

In [ ]:
# CELL 6 — Table 1: Architecture Summary
# Extracts: Total Parameters, GFLOPs, Layer Count as required by brief Section 6.1
print("\n" + "="*65)
print("TABLE 1: YOLOv10-M Architecture Summary")
print("="*65)

try:
    import contextlib, io as _io
    arch_model = YOLO(MODEL_WEIGHTS)

    # Capture model.info() output to extract GFLOPs & layer count
    f_buf = _io.StringIO()
    with contextlib.redirect_stdout(f_buf):
        arch_model.info(verbose=True)
    info_text = f_buf.getvalue()

    # Parse parameters, GFLOPs, layers from the info string
    total_params     = sum(p.numel() for p in arch_model.model.parameters())
    trainable_params = sum(p.numel() for p in arch_model.model.parameters() if p.requires_grad)

    # Attempt to extract GFLOPs and layer count from info string
    gflops_val     = "N/A"
    layer_count_val = "N/A"
    for line in info_text.splitlines():
        line_l = line.lower()
        if "gflop" in line_l:
            import re
            nums = re.findall(r"[\d]+\.?[\d]*", line)
            if nums:
                gflops_val = f"{float(nums[-1]):.1f} GFLOPs"
        if "layer" in line_l:
            import re
            nums = re.findall(r"[\d]+", line)
            if nums:
                layer_count_val = nums[0]

    # Fallback: try model.info() tuple return (some Ultralytics versions)
    try:
        info_tuple = arch_model.model.info()
        if isinstance(info_tuple, (list, tuple)) and len(info_tuple) >= 4:
            layer_count_val = str(info_tuple[0])
            total_params    = int(info_tuple[1])
            trainable_params= int(info_tuple[2])
            gflops_val      = f"{info_tuple[3]:.1f} GFLOPs"
    except Exception:
        pass

    arch_rows = [
        ("Model Variant",        "YOLOv10-M (Medium)"),
        ("Layer Count",          layer_count_val),
        ("Total Parameters",     f"{total_params:,}"),
        ("Trainable Parameters", f"{trainable_params:,}"),
        ("GFLOPs",               gflops_val),
        ("Pretrained On",        "COCO (80 classes)"),
        ("Input Resolution",     "640 × 640 px"),
        ("Optimizer",            "AdamW"),
        ("Batch Size",           "16"),
        ("Training Epochs",      "100"),
        ("Augmentations",        "Mosaic (1.0)  |  MixUp (0.15)  |  CutMix (0.10)"),
        ("LRs Evaluated",        "  |  ".join(str(lr) for lr in LEARNING_RATES)),
        ("Number of Classes",    str(len(CLASS_NAMES))),
        ("Classes",              ", ".join(CLASS_NAMES)),
        ("Framework",            f"Ultralytics {__import__('ultralytics').__version__}"),
        ("Training Hardware",    "Kaggle Tesla T4 GPU"),
        ("Random Seed",          str(SEED)),
    ]
    table1_df = pd.DataFrame(arch_rows, columns=["Attribute", "Value"])
    print(table1_df.to_string(index=False))

    # ── Save CSV ──────────────────────────────────────────────
    table1_df.to_csv(OUTPUT_DIR / "Group8_Table1_ModelArchitecture.csv", index=False)

    # ── Save styled PNG ───────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, len(arch_rows) * 0.52 + 1.2))
    ax.axis("off")
    tbl = ax.table(
        cellText   = table1_df.values,
        colLabels  = table1_df.columns,
        cellLoc    = "left",
        loc        = "center",
        colWidths  = [0.38, 0.58],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.55)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor("#1565C0")
            cell.set_text_props(color="white", fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#E3F2FD")
        else:
            cell.set_facecolor("white")
        cell.set_edgecolor("#BBDEFB")
    ax.set_title("Table 1: YOLOv10-M Architecture Summary",
                 fontsize=13, fontweight="bold", pad=18, loc="left")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "Group8_Table1_ModelArchitecture.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("\nTable 1 saved ✓")

    del arch_model
    torch.cuda.empty_cache()

except Exception as e:
    print(f"Table 1 generation failed: {e}")
    traceback.print_exc()
    plt.close("all")


---
### Cell 7 — Table 2: Learning Rate Comparison

In [ ]:
# CELL 7 — Table 2: Optimal Learning Rate Selection
# Required columns (per brief): LR | mAP@0.5 | mAP@0.5:0.95 | Precision | Recall
# Best-performing row is highlighted in yellow.
print("\n" + "="*65)
print("TABLE 2: Learning Rate Comparison")
print("="*65)

try:
    summary_rows = []
    for lr in LEARNING_RATES:
        if lr not in best_lr_per_run:
            continue
        m = best_lr_per_run[lr]
        summary_rows.append({
            "LR"           : lr,
            "Best Epoch"   : m["epoch"],
            "mAP@0.5"      : round(m["mAP50"],     4),
            "mAP@0.5:0.95" : round(m["mAP50_95"],  4),
            "Precision"    : round(m["precision"],  4),
            "Recall"       : round(m["recall"],     4),
        })
    table2_df = pd.DataFrame(summary_rows)
    print(table2_df.to_string(index=False))
    print(f"\nBest LR: {BEST_LR}  (mAP@0.5 = {best_lr_per_run[BEST_LR]['mAP50']:.4f})")

    table2_df.to_csv(OUTPUT_DIR / "Group8_Table2_LRComparison.csv", index=False)

    # ── Styled PNG ────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(13, len(summary_rows) * 0.55 + 1.4))
    ax.axis("off")
    tbl = ax.table(
        cellText   = table2_df.values,
        colLabels  = table2_df.columns,
        cellLoc    = "center",
        loc        = "center",
        colWidths  = [0.14, 0.14, 0.16, 0.19, 0.16, 0.16],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1, 1.85)
    best_row_idx = table2_df["mAP@0.5"].idxmax() + 1   # +1 for header row
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor("#1565C0")
            cell.set_text_props(color="white", fontweight="bold")
        elif r == best_row_idx:
            cell.set_facecolor("#FFF9C4")
            cell.set_text_props(fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#E3F2FD")
        else:
            cell.set_facecolor("white")
        cell.set_edgecolor("#BBDEFB")
    ax.set_title(
        f"Table 2: Optimal Learning Rate Selection  (best row highlighted) — Best LR = {BEST_LR}",
        fontsize=12, fontweight="bold", pad=16, loc="left",
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "Group8_Table2_LRComparison.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Table 2 saved ✓")

except Exception as e:
    print(f"Table 2 generation failed: {e}")
    traceback.print_exc()
    plt.close("all")


---
### Cell 8 — Figure 2: Multi-LR Convergence Graph

In [ ]:
# CELL 8 — Figure 2: Validation mAP@0.5 across epochs for all 5 LRs
# X-axis = Epochs | Y-axis = Val mAP@0.5 | 5 lines | legend required
print("\nGenerating Figure 2 — LR Convergence Plot …")

try:
    fig, ax = plt.subplots(figsize=(14, 7))

    for i, lr in enumerate(LEARNING_RATES):
        if lr not in all_results:
            continue
        df     = all_results[lr]
        epochs = df["epoch"] + 1
        mAP50  = df["metrics/mAP50(B)"]
        ax.plot(
            epochs, mAP50,
            label      = LR_LABELS[i],
            color      = LR_COLOURS[i],
            linewidth  = 2.2,
            marker     = LR_MARKERS[i],
            markevery  = 10,
            markersize = 6,
            alpha      = 0.92,
            zorder     = 3,
        )

    # ── Star marker on best LR peak ──────────────────────────
    if BEST_LR in all_results:
        best_df  = all_results[BEST_LR]
        peak_idx = best_df["metrics/mAP50(B)"].idxmax()
        peak_ep  = int(peak_idx) + 1
        peak_val = float(best_df.loc[peak_idx, "metrics/mAP50(B)"])
        ax.scatter(peak_ep, peak_val, s=320, c="#FFD600", marker="*",
                   zorder=6, edgecolors="black", linewidths=0.8,
                   label=f"Peak: LR={BEST_LR}  mAP={peak_val:.3f} (ep {peak_ep})")
        ax.annotate(
            f" mAP = {peak_val:.3f}\n Epoch {peak_ep}",
            xy=(peak_ep, peak_val),
            xytext=(min(peak_ep + 5, 90), max(peak_val - 0.09, 0.05)),
            fontsize=9.5, color="#333333",
            arrowprops=dict(arrowstyle="->", color="#555555", lw=1.2),
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#CCCCCC", alpha=0.88),
            zorder=7,
        )

    ax.set_xlabel("Epoch",                 fontsize=13, fontweight="bold")
    ax.set_ylabel("Validation mAP@0.5",   fontsize=13, fontweight="bold")
    ax.set_title(
        "Figure 2: YOLOv10-M — Validation mAP@0.5 Across Learning Rates (100 Epochs)",
        fontsize=14, fontweight="bold", pad=16,
    )
    ax.set_xlim(1, 100)
    ax.set_ylim(bottom=0, top=min(all_results[list(all_results.keys())[0]]["metrics/mAP50(B)"].max() * 1.15 + 0.05, 1.05))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(0.1))
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.tick_params(labelsize=11)
    ax.legend(
        fontsize=10.5, loc="upper left",
        bbox_to_anchor=(1.01, 1.0), borderaxespad=0,
        framealpha=0.95, edgecolor="#CCCCCC",
        title="Experiment", title_fontsize=11,
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "Group8_Figure2_LRConvergence.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / "Group8_Figure2_LRConvergence.pdf", bbox_inches="tight")
    plt.close(fig)
    print("Figure 2 saved ✓")

except Exception as e:
    print(f"Figure 2 generation failed: {e}")
    traceback.print_exc()
    plt.close("all")


---
### Cell 9 — Table 3: Per-Class Performance (best LR, test set)

In [ ]:
# CELL 9 — Table 3 & Figure 4: Per-Class Performance at best LR
# Required columns: Class | Precision | Recall | mAP@0.5
# Evaluated on test split (falls back to val if test unavailable)
print("\nEvaluating best model on test/val set …")

best_model = None
table3_full = None

try:
    best_weights = LR_FOLDER_MAP[BEST_LR] / "weights" / "best.pt"
    assert best_weights.exists(), f"best.pt not found at {best_weights}"

    best_model = YOLO(str(best_weights))

    eval_split = (
        "test"
        if img_dir("test").exists() and dataset_counts.get("test", 0) > 0
        else "val"
    )
    print(f"  Evaluating on: {eval_split} split ({dataset_counts.get(eval_split, 0)} images)")

    test_results = best_model.val(
        data     = str(DATA_YAML),
        split    = eval_split,
        imgsz    = 640,
        batch    = 8,
        device   = DEVICE,
        plots    = True,
        save_json= True,
        project  = str(OUTPUT_DIR),
        name     = "test_evaluation",
        exist_ok = True,
        verbose  = False,
    )
    print("  Evaluation complete.")

    # ── Per-class metrics ─────────────────────────────────────
    class_metrics = []
    for i, cls_name in enumerate(CLASS_NAMES):
        try:
            res = test_results.box.class_result(i)
            p, r, ap50, ap50_95 = float(res[0]), float(res[1]), float(res[2]), float(res[3])
        except Exception:
            p, r, ap50, ap50_95 = 0.0, 0.0, 0.0, 0.0
        class_metrics.append({
            "Class"        : cls_name,
            "Precision"    : round(p,      4),
            "Recall"       : round(r,      4),
            "mAP@0.5"      : round(ap50,   4),
            "mAP@0.5:0.95" : round(ap50_95, 4),
        })

    table3_df = pd.DataFrame(class_metrics)
    overall_row = pd.DataFrame([{
        "Class"        : "Overall",
        "Precision"    : round(float(test_results.box.mp),    4),
        "Recall"       : round(float(test_results.box.mr),    4),
        "mAP@0.5"      : round(float(test_results.box.map50), 4),
        "mAP@0.5:0.95" : round(float(test_results.box.map),   4),
    }])
    table3_full = pd.concat([table3_df, overall_row], ignore_index=True)

    print("\n" + "="*65)
    print(f"TABLE 3: Per-Class Performance  (LR={BEST_LR}, {eval_split} set)")
    print("="*65)
    print(table3_full.to_string(index=False))

    table3_full.to_csv(OUTPUT_DIR / "Group8_Table3_PerClassPerformance.csv", index=False)

    # ── Styled PNG ────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(11, len(table3_full) * 0.55 + 1.4))
    ax.axis("off")
    tbl = ax.table(
        cellText  = table3_full.values,
        colLabels = table3_full.columns,
        cellLoc   = "center",
        loc       = "center",
        colWidths = [0.20, 0.18, 0.18, 0.18, 0.22],
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.scale(1, 1.85)
    overall_row_idx = len(table3_full)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor("#1565C0")
            cell.set_text_props(color="white", fontweight="bold")
        elif r == overall_row_idx:
            cell.set_facecolor("#E8F5E9")
            cell.set_text_props(fontweight="bold")
        elif r % 2 == 0:
            cell.set_facecolor("#E3F2FD")
        else:
            cell.set_facecolor("white")
        cell.set_edgecolor("#BBDEFB")
    ax.set_title(
        f"Table 3: Per-Class Detection Performance  (Best LR = {BEST_LR} | {eval_split} set)",
        fontsize=12, fontweight="bold", pad=16, loc="left",
    )
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "Group8_Table3_PerClassPerformance.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Table 3 saved ✓")

    # ── Figure 4: Per-Class Bar Chart ─────────────────────────
    metrics_plot = ["Precision", "Recall", "mAP@0.5", "mAP@0.5:0.95"]
    bar_clrs     = ["#1565C0", "#2E7D32", "#E65100", "#6A1B9A"]

    fig, axes = plt.subplots(1, 4, figsize=(19, 5.5), sharey=True)
    fig.suptitle(
        f"Figure 4: YOLOv10-M Per-Class Detection Performance  (Best LR = {BEST_LR})",
        fontsize=14, fontweight="bold", y=1.02,
    )
    for idx, (metric, clr) in enumerate(zip(metrics_plot, bar_clrs)):
        ax_   = axes[idx]
        vals  = table3_df[metric].values
        bars  = ax_.bar(CLASS_NAMES, vals, color=clr, edgecolor="white",
                        linewidth=1.2, alpha=0.88, width=0.55)
        for bar in bars:
            h = bar.get_height()
            ax_.text(bar.get_x() + bar.get_width() / 2, h + 0.015,
                     f"{h:.3f}", ha="center", va="bottom",
                     fontsize=10.5, fontweight="bold", color="#222222")
        ax_.set_title(metric,  fontsize=12, fontweight="bold", pad=10)
        ax_.set_ylim(0, 1.12)
        ax_.set_ylabel("Score" if idx == 0 else "", fontsize=11)
        ax_.yaxis.set_major_locator(ticker.MultipleLocator(0.2))
        ax_.grid(axis="y", alpha=0.25, linestyle="--")
        ax_.tick_params(axis="x", labelsize=10.5)
        ax_.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "Group8_Figure4_PerClassMetrics.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Figure 4 saved ✓")

except Exception as e:
    print(f"Table 3 / Figure 4 failed: {e}")
    traceback.print_exc()
    plt.close("all")
    best_model = None


---
### Cell 10 — Figure 3: Qualitative Results (6 images)

In [ ]:
# CELL 10 — Figure 3: Qualitative Detection Samples
# Brief requires 6 sample images including 2 correct detections in dense scenes
print("\nGenerating Figure 3 — Qualitative Results …")

try:
    if best_model is None:
        best_weights = LR_FOLDER_MAP[BEST_LR] / "weights" / "best.pt"
        best_model   = YOLO(str(best_weights))

    inf_split   = (
        "test"
        if img_dir("test").exists() and dataset_counts.get("test", 0) > 0
        else "val"
    )
    sample_pool = sorted([
        f for f in img_dir(inf_split).iterdir()
        if f.suffix.lower() in IMG_EXTS
    ])
    print(f"  Image pool: {len(sample_pool)} images from '{inf_split}' split")

    # ── Stratified selection ──────────────────────────────────
    # Priority: (1) multi-object/dense scenes, (2) one image per class
    images_by_class   = defaultdict(list)
    multi_object_imgs = []

    for img_path in sample_pool:
        lbl_path = lbl_dir(inf_split) / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue
        with open(lbl_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        cls_set = {int(l.split()[0]) for l in lines}
        for cid in cls_set:
            images_by_class[cid].append(img_path)
        if len(lines) >= 3:
            multi_object_imgs.append(img_path)

    selected = []
    # 2 dense / multi-object images first
    for img in multi_object_imgs[:2]:
        if img not in selected:
            selected.append(img)
    # One representative per class
    for cid in range(len(CLASS_NAMES)):
        for img in images_by_class.get(cid, []):
            if img not in selected and len(selected) < 6:
                selected.append(img)
    # Pad with remaining images if needed
    for img in sample_pool:
        if img not in selected and len(selected) < 6:
            selected.append(img)
    selected = selected[:6]
    print(f"  Selected {len(selected)} images for Figure 3")

    # ── 2×3 detection grid ────────────────────────────────────
    fig = plt.figure(figsize=(20, 13))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.10, wspace=0.06)

    for idx, img_path in enumerate(selected):
        try:
            result = best_model.predict(
                source  = str(img_path),
                imgsz   = 640,
                conf    = 0.25,
                device  = DEVICE,
                verbose = False,
            )[0]
            annotated     = result.plot(line_width=2, font_size=10)
            annotated_rgb = annotated[:, :, ::-1]   # BGR → RGB
            n_det         = len(result.boxes) if result.boxes is not None else 0
        except Exception as pred_err:
            print(f"  Prediction failed for {img_path.name}: {pred_err}")
            import numpy as np
            annotated_rgb = np.zeros((640, 640, 3), dtype=np.uint8)
            n_det         = 0

        ax = fig.add_subplot(gs[idx // 3, idx % 3])
        ax.imshow(annotated_rgb)
        ax.set_title(
            f"Sample {idx+1}  |  {n_det} detection{'s' if n_det != 1 else ''}",
            fontsize=11, fontweight="bold", pad=6,
        )
        ax.axis("off")

    fig.suptitle(
        f"Figure 3: YOLOv10-M Qualitative Detection Results  (Best LR = {BEST_LR}, conf ≥ 0.25)",
        fontsize=15, fontweight="bold", y=1.01,
    )
    fig.savefig(OUTPUT_DIR / "Group8_Figure3_QualitativeResults.png",
                dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("Figure 3 saved ✓")

except Exception as e:
    print(f"Figure 3 failed: {e}")
    traceback.print_exc()
    plt.close("all")


---
### Cell 11 — FPS / Inference Speed Benchmark

In [ ]:
# CELL 11 — FPS Inference Speed Benchmark
# Required metric (brief Section 5e): FPS on standard GPU (T4 / V100)
print("\nRunning FPS Benchmark …")

avg_fps = std_fps = avg_lat = hw_name = None

try:
    if best_model is None:
        best_model = YOLO(str(LR_FOLDER_MAP[BEST_LR] / "weights" / "best.pt"))

    bench_split = (
        "test"
        if img_dir("test").exists() and dataset_counts.get("test", 0) > 0
        else "val"
    )
    bench_images = [
        str(f) for f in img_dir(bench_split).iterdir()
        if f.suffix.lower() in IMG_EXTS
    ][:50]
    print(f"  Benchmark images: {len(bench_images)} from '{bench_split}' split")

    # GPU warm-up (3 forward passes)
    for _ in range(3):
        best_model.predict(bench_images[0], imgsz=640, device=DEVICE, verbose=False)

    ROUNDS = 3
    fps_vals, lat_vals = [], []
    for rnd in range(ROUNDS):
        t0 = time.perf_counter()
        for img in bench_images:
            best_model.predict(img, imgsz=640, device=DEVICE, verbose=False)
        elapsed = time.perf_counter() - t0
        fps     = len(bench_images) / elapsed
        lat     = 1000 * elapsed / len(bench_images)
        fps_vals.append(fps)
        lat_vals.append(lat)
        print(f"  Round {rnd+1}: {elapsed:.2f}s → {fps:.1f} FPS  ({lat:.1f} ms/img)")

    avg_fps  = float(np.mean(fps_vals))
    std_fps  = float(np.std(fps_vals))
    avg_lat  = float(np.mean(lat_vals))
    hw_name  = (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else platform.processor()
    )

    print(f"\n  Average FPS     : {avg_fps:.1f} ± {std_fps:.1f}")
    print(f"  Average Latency : {avg_lat:.1f} ms/image")
    print(f"  Hardware        : {hw_name}")

    bench_txt = (
        f"Model           : YOLOv10-M\n"
        f"Best LR         : {BEST_LR}\n"
        f"Image Size      : 640 x 640 px\n"
        f"Hardware        : {hw_name}\n"
        f"Benchmark Split : {bench_split} ({len(bench_images)} images)\n"
        f"Rounds          : {ROUNDS}\n"
        f"Average FPS     : {avg_fps:.1f} +/- {std_fps:.1f}\n"
        f"Average Latency : {avg_lat:.1f} ms/image\n"
        f"Timestamp       : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n"
    )
    (OUTPUT_DIR / "Group8_FPS_Benchmark.txt").write_text(bench_txt)
    print("FPS Benchmark saved ✓")

    torch.cuda.empty_cache()

except Exception as e:
    print(f"FPS Benchmark failed: {e}")
    traceback.print_exc()


---
### Cell 12 — Final Report Data Summary

In [ ]:
# CELL 12 — Final Summary Print (all numbers needed for report writing)
print("\n" + "="*70)
print("FINAL REPORT DATA SUMMARY — GROUP 8  |  MCE 415")
print("="*70)
print(f"\n  Model     : YOLOv10-M (COCO pretrained)")
print(f"  Best LR   : {BEST_LR}")
print(f"  Best mAP@0.5 : {best_lr_per_run[BEST_LR]['mAP50']:.4f}")
print(f"\n  Dataset Split:")
for split, n in dataset_counts.items():
    if n > 0:
        print(f"    {split:>5}: {n} images")
print(f"    Total: {total_images} images | Classes: {', '.join(CLASS_NAMES)}")

print("\n  TABLE 2 — LR Comparison:")
try:
    print(table2_df.to_string(index=False))
except NameError:
    print("  (not available)")

print("\n  TABLE 3 — Per-Class Performance:")
try:
    print(table3_full.to_string(index=False))
except (NameError, TypeError):
    print("  (not available — evaluation may have failed)")

if avg_fps is not None:
    print(f"\n  FPS     : {avg_fps:.1f} ± {std_fps:.1f}")
    print(f"  Latency : {avg_lat:.1f} ms/image")
    print(f"  Hardware: {hw_name}")

print("\n  Output files generated:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print(f"    {f.relative_to(OUTPUT_DIR)}  ({f.stat().st_size / 1024:.0f} KB)")


---
### Cell 13 — Package all outputs into a zip for download

In [ ]:
# CELL 13 — Bundle Group8_Results into a single zip for Kaggle Output tab
print("\nPackaging outputs …")

zip_out = Path("/kaggle/working/Group8_YOLOv10_Analysis_Results.zip")
try:
    with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
        for fpath in sorted(OUTPUT_DIR.rglob("*")):
            if fpath.is_file():
                arcname = fpath.relative_to(OUTPUT_DIR.parent)
                zf.write(fpath, arcname)
    size_mb = zip_out.stat().st_size / (1024 * 1024)
    print(f"  Zip : {zip_out}  ({size_mb:.1f} MB)")
    print("\nDone! Download  Group8_YOLOv10_Analysis_Results.zip  from the Kaggle Output tab.")
    print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
except Exception as e:
    print(f"Packaging failed: {e}")
    traceback.print_exc()
